# PCA for Early Deterioration Scoring

## From 15 Bedside Channels to 1 Early Warning Signal

Welcome to this executive demonstration of **Principal Component Analysis (PCA)** - a technique that turns an overwhelming wall of bedside monitor data into a single, actionable alert score.

**What you'll see:**
- Why watching 15+ channels per bed doesn't scale
- How PCA compresses monitoring data while preserving patterns
- The key insight: **reconstruction error as an early warning**
- Ward-wide monitoring with one number per bed

> **A note on the data.** Every patient in this notebook is simulated from published
> physiological ranges. No real or identifiable patient data is used.

---


In [1]:
# Core imports for data analysis and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Interactive widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, RadioButtons, HBox, VBox, Output
from IPython.display import display, HTML

# Set random seed for reproducibility
np.random.seed(42)

print("Ready for PCA Executive Demonstration")
print("Application: Early Deterioration Scoring in Critical Care")


Ready for PCA Executive Demonstration
Application: Early Deterioration Scoring in Critical Care


---

## 1. The $50,000 Question: Which Channels Actually Matter?

**The Challenge Your Critical Care Team Faces Every Day:**

Every ICU bed streams 100+ measurements. Nurses and intensivists see thousands of numbers per shift:
- Hemodynamics: heart rate, arterial pressure, central venous pressure
- Respiratory: SpO2, respiratory rate, end-tidal CO2, ventilator settings
- Laboratory: lactate, glucose, renal markers
- Neurological and comfort: conscious level, sedation, pain

**The Problem:**
- Alarm fatigue: most monitor alarms are not clinically actionable, so real ones get muted
- A patient can deteriorate while every single channel is still inside its own limits
- Staff spend the shift watching screens instead of at the bedside
- Deterioration is recognized late, when rescue is harder and costlier

**The Promise:**
What if you could watch an entire ward using just **3-5 "vital signs of the vital signs"** that capture **95% of what matters**?


In [2]:
# Visualize the monitoring problem
def create_dashboard_comparison():
    """Show the before/after of bedside monitoring."""

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            'BEFORE: Current Monitoring (Overwhelming)',
            'AFTER: PCA-Powered Dashboard (Actionable)'
        ),
        horizontal_spacing=0.12
    )

    # LEFT SIDE: Overwhelming channel display
    channel_names = [
        'Heart Rate', 'MAP', 'SpO2', 'Resp Rate', 'Temp',
        'EtCO2', 'CVP', 'Urine Out', 'Lactate', 'Glucose',
        'GCS', 'FiO2', 'PEEP', 'Tidal Vol', 'Pain'
    ]

    np.random.seed(42)
    x_positions = np.tile(np.arange(5), 3)
    y_positions = np.repeat(np.arange(3), 5)
    values = np.random.uniform(30, 95, 15)
    colors = ['#e74c3c' if v > 80 else '#f39c12' if v > 60 else '#2ecc71' for v in values]

    fig.add_trace(
        go.Scatter(
            x=x_positions,
            y=y_positions,
            mode='markers+text',
            marker=dict(size=50, color=colors, line=dict(color='black', width=2)),
            text=[f'{v:.0f}%' for v in values],
            textposition='middle center',
            textfont=dict(size=10, color='white'),
            hovertext=channel_names,
            hoverinfo='text',
            showlegend=False
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=x_positions,
            y=y_positions - 0.35,
            mode='text',
            text=channel_names,
            textposition='middle center',
            textfont=dict(size=8, color='black'),
            showlegend=False
        ),
        row=1, col=1
    )

    # RIGHT SIDE: Single deviation score (the real value of PCA)
    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[1],
            mode='markers+text',
            marker=dict(size=180, color='#2ecc71', line=dict(color='black', width=3)),
            text=['STABLE'],
            textposition='middle center',
            textfont=dict(size=24, color='white', family='Arial Black'),
            showlegend=False
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[0.2],
            mode='text',
            text=['Deviation Score: 0.3'],
            textposition='middle center',
            textfont=dict(size=14, color='black'),
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(text='The Monitoring Problem: Before and After PCA', font=dict(size=20)),
        width=1200,
        height=450,
        showlegend=False
    )

    for col in [1, 2]:
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col)

    fig.show()

    print("\n15 channels → 1 deviation score")
    print("Stable patient = low score, deteriorating patient = high score")
    print("Simple. Actionable. Early warning.")

create_dashboard_comparison()



15 channels → 1 deviation score
Stable patient = low score, deteriorating patient = high score
Simple. Actionable. Early warning.


---

## 2. The Concept: Finding the Best Viewing Angle

**A Simple Analogy:**

Imagine you're photographing a 3D sculpture. Different camera angles capture different amounts of detail:
- From one angle, you see just a flat silhouette
- From another, you see rich depth and detail

**PCA does the same thing with data:**
- Your channels create a "sculpture" in high-dimensional space
- PCA finds the "viewing angle" that shows you the most information
- Each principal component is like looking from a different direction

**Let's see this in action with a 3D example...**

In [3]:
def create_3d_shadow_demo():
    """Interactive 3D demonstration of the 'shadow' concept."""
    
    # Generate 3D data that has a clear principal direction
    np.random.seed(42)
    n_points = 200
    
    # Create elongated cluster (like a cigar shape)
    # Main axis of variation
    t = np.random.uniform(-3, 3, n_points)
    
    # Add some spread in perpendicular directions
    x = t * 0.8 + np.random.normal(0, 0.3, n_points)
    y = t * 0.6 + np.random.normal(0, 0.3, n_points)
    z = t * 0.4 + np.random.normal(0, 0.3, n_points)
    
    # Fit PCA to find principal directions
    data_3d = np.column_stack([x, y, z])
    pca = PCA(n_components=3)
    pca.fit(data_3d)
    
    # Get principal direction
    pc1_direction = pca.components_[0] * 3  # Scale for visibility
    
    # Create figure
    fig = go.Figure()
    
    # Add 3D scatter points
    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(
                size=5,
                color=t,
                colorscale='Viridis',
                opacity=0.7
            ),
            name='channel Data'
        )
    )
    
    # Add principal direction arrow
    fig.add_trace(
        go.Scatter3d(
            x=[0, pc1_direction[0]],
            y=[0, pc1_direction[1]],
            z=[0, pc1_direction[2]],
            mode='lines',
            line=dict(color='red', width=8),
            name='Main Direction (PC1)'
        )
    )
    
    # Add projections onto the XY plane ("shadow")
    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=np.full_like(z, z.min() - 0.5),
            mode='markers',
            marker=dict(
                size=3,
                color='gray',
                opacity=0.3
            ),
            name='Projection (Shadow)'
        )
    )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='The Shadow Analogy: Finding the Best Viewing Angle<br><sub>Rotate this view to see how different angles capture different amounts of information</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis_title='channel A',
            yaxis_title='channel B',
            zaxis_title='channel C',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=1000,
        height=700,
        showlegend=True
    )
    
    fig.show()
    
    # Explain the variance
    print("\nVariance explained by each direction:")
    print(f"  Main Direction (PC1): {pca.explained_variance_ratio_[0]*100:.1f}%")
    print(f"  Second Direction (PC2): {pca.explained_variance_ratio_[1]*100:.1f}%")
    print(f"  Third Direction (PC3): {pca.explained_variance_ratio_[2]*100:.1f}%")
    print("\nThe red arrow shows the direction that captures the most variation.")
    print("This is the 'best viewing angle' for understanding this data.")

create_3d_shadow_demo()


Variance explained by each direction:
  Main Direction (PC1): 95.3%
  Second Direction (PC2): 2.6%
  Third Direction (PC3): 2.1%

The red arrow shows the direction that captures the most variation.
This is the 'best viewing angle' for understanding this data.


In [4]:
def interactive_projection_demo(viewing_angle=45):
    """Show how different viewing angles capture different amounts of spread."""
    
    # Generate elongated data
    np.random.seed(42)
    n_points = 150
    t = np.random.uniform(-3, 3, n_points)
    x = t + np.random.normal(0, 0.3, n_points)
    y = t * 0.3 + np.random.normal(0, 0.3, n_points)
    
    # Calculate projection onto a line at the given angle
    angle_rad = np.radians(viewing_angle)
    direction = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    
    # Project points onto line
    data = np.column_stack([x, y])
    projections = data @ direction
    projected_points = np.outer(projections, direction)
    
    # Calculate spread (variance) of projections
    spread = np.var(projections)
    max_spread = np.var(data @ np.array([0.95, 0.31]))  # Approximate optimal
    
    # Create figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Viewing Angle: {viewing_angle}°',
            'Information Captured'
        ),
        column_widths=[0.7, 0.3]
    )
    
    # Plot original data
    fig.add_trace(
        go.Scatter(
            x=x, y=y,
            mode='markers',
            marker=dict(size=8, color='lightblue', opacity=0.6),
            name='Data',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Plot projection line
    line_extent = 5
    fig.add_trace(
        go.Scatter(
            x=[-line_extent * direction[0], line_extent * direction[0]],
            y=[-line_extent * direction[1], line_extent * direction[1]],
            mode='lines',
            line=dict(color='red', width=3),
            name='Viewing Direction',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Plot projected points
    fig.add_trace(
        go.Scatter(
            x=projected_points[:, 0],
            y=projected_points[:, 1],
            mode='markers',
            marker=dict(size=6, color='red', opacity=0.5),
            name='Shadow',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Information captured gauge
    info_captured = (spread / max_spread) * 100
    info_captured = min(info_captured, 100)  # Cap at 100%
    
    fig.add_trace(
        go.Bar(
            x=['Info Captured'],
            y=[info_captured],
            marker_color='green' if info_captured > 80 else 'orange' if info_captured > 50 else 'red',
            text=[f'{info_captured:.0f}%'],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title='Finding the Best Viewing Angle',
        width=1000,
        height=450
    )
    
    fig.update_xaxes(range=[-4, 4], row=1, col=1)
    fig.update_yaxes(range=[-2, 2], row=1, col=1)
    fig.update_yaxes(range=[0, 110], row=1, col=2)
    
    fig.show()

# Create interactive widget
interact(
    interactive_projection_demo,
    viewing_angle=IntSlider(
        value=45,
        min=0,
        max=180,
        step=5,
        description='Viewing Angle:',
        style={'description_width': '100px'}
    )
);

interactive(children=(IntSlider(value=45, description='Viewing Angle:', max=180, step=5, style=SliderStyle(des…

**Key Insight:**

Notice how some viewing angles show wide spread (lots of information) while others show points bunched together (information lost). PCA automatically finds the angle that maximizes spread - giving you the most information in the fewest dimensions.

---

## 3. Real-World Scenario: Monitoring an ICU Bed

Let's apply this to bedside monitoring data. We'll work with 15 channels routinely charted for a ventilated ICU patient:

| Category | Channels |
|----------|--------|
| **Hemodynamic** | Heart rate, Mean arterial pressure, Central venous pressure, Urine output |
| **Respiratory** | SpO2, Respiratory rate, End-tidal CO2, FiO2, PEEP, Tidal volume |
| **Metabolic** | Temperature, Lactate, Glucose |
| **Neurological / comfort** | Glasgow Coma Scale, Pain score |


In [5]:
def generate_icu_channel_data(n_samples=500, include_anomalies=True):
    """
    Generate realistic synthetic ICU bedside data with natural correlations.

    Channels move together because a small number of hidden clinical factors
    drive them all:
    - illness_severity : how sick the patient is right now
    - sedation_depth   : how deeply sedated they are
    - age_frailty      : physiological reserve

    Values are simulated from published physiological ranges. No real or
    identifiable patient data is used.
    """
    np.random.seed(42)

    # Hidden clinical factors that drive the correlations
    illness_severity = np.random.uniform(0.0, 1.0, n_samples)   # primary driver
    sedation_depth = np.random.uniform(0.0, 1.0, n_samples)     # sedation / analgesia
    age_frailty = np.linspace(0, 1, n_samples)                  # reserve, low to high

    # Hemodynamic channels
    # Tachycardia is the earliest and most reliable sign of physiological stress
    heart_rate = 72 + 46 * illness_severity - 12 * sedation_depth + 6 * age_frailty \
        + np.random.normal(0, 2.4, n_samples)
    # Mean arterial pressure falls as the patient deteriorates
    map_mmhg = 88 - 24 * illness_severity - 9 * sedation_depth + np.random.normal(0, 2.4, n_samples)
    # Filling pressure rises with fluid loading and right heart strain
    cvp = 7 + 6 * illness_severity + 1.5 * age_frailty + np.random.normal(0, 0.7, n_samples)
    # Urine output falls first - the kidney is an early casualty of poor perfusion
    urine_output = 1.1 - 0.75 * illness_severity - 0.2 * age_frailty + np.random.normal(0, 0.07, n_samples)

    # Respiratory channels
    spo2 = 98 - 9 * illness_severity - 1.0 * sedation_depth + np.random.normal(0, 0.6, n_samples)
    resp_rate = 14 + 17 * illness_severity - 5 * sedation_depth + np.random.normal(0, 1.1, n_samples)
    # Deep sedation means hypoventilation and CO2 retention; sepsis blows CO2 off
    etco2 = 38 - 7 * illness_severity + 6 * sedation_depth + np.random.normal(0, 1.2, n_samples)
    fio2 = 0.28 + 0.48 * illness_severity + np.random.normal(0, 0.018, n_samples)
    peep = 5 + 8 * illness_severity + np.random.normal(0, 0.5, n_samples)
    tidal_volume = 460 - 70 * illness_severity + np.random.normal(0, 15, n_samples)

    # Metabolic channels
    temp_c = 36.8 + 1.5 * illness_severity - 0.3 * sedation_depth + np.random.normal(0, 0.15, n_samples)
    # Lactate is the classic marker of tissue hypoperfusion
    lactate = 0.9 + 5.0 * illness_severity + np.random.normal(0, 0.25, n_samples)
    glucose = 6.1 + 3.05 * illness_severity + 0.67 * age_frailty + np.random.normal(0, 0.4, n_samples)  # mmol/L

    # Neurological / comfort channels
    gcs = 15 - 7 * sedation_depth - 3.5 * illness_severity + np.random.normal(0, 0.35, n_samples)
    pain_score = 3 + 4 * illness_severity - 3.5 * sedation_depth + np.random.normal(0, 0.42, n_samples)

    # Create DataFrame
    df = pd.DataFrame({
        'heart_rate': heart_rate,
        'map_mmhg': map_mmhg,
        'spo2': spo2,
        'resp_rate': resp_rate,
        'temp_c': temp_c,
        'etco2': etco2,
        'cvp': cvp,
        'urine_output': urine_output,
        'lactate': lactate,
        'glucose': glucose,
        'gcs': gcs,
        'fio2': fio2,
        'peep': peep,
        'tidal_volume': tidal_volume,
        'pain_score': pain_score
    })

    # Add some anomalies if requested
    if include_anomalies:
        # 3% of readings are acute deterioration events
        n_anomalies = int(n_samples * 0.03)
        anomaly_indices = np.random.choice(n_samples, n_anomalies, replace=False)

        # A real alert is a DECOUPLING: channels that normally move together
        # stop doing so. Here the heart rate jumps and the pressure falls while
        # temperature, lactate, FiO2 and the rest sit still - the picture of a
        # new fast arrhythmia or an acute bleed, not of a patient who is simply
        # sicker. PCA has never seen that combination, so it reconstructs badly.
        df.loc[anomaly_indices, 'heart_rate'] += np.random.uniform(30, 45, n_anomalies)
        df.loc[anomaly_indices, 'map_mmhg'] -= np.random.uniform(12, 20, n_anomalies)

        df['is_anomaly'] = False
        df.loc[anomaly_indices, 'is_anomaly'] = True
    else:
        df['is_anomaly'] = False

    # Keep every value inside a physiologically possible range
    limits = {
        'heart_rate': (40, 180), 'map_mmhg': (40, 130), 'spo2': (70, 100),
        'resp_rate': (6, 45), 'temp_c': (34.0, 41.5), 'etco2': (15, 60),
        'cvp': (0, 25), 'urine_output': (0.0, 4.0), 'lactate': (0.4, 15.0),
        'glucose': (3.0, 25.0), 'gcs': (3, 15), 'fio2': (0.21, 1.0),
        'peep': (0, 20), 'tidal_volume': (250, 650), 'pain_score': (0, 10),
    }
    for col, (lo, hi) in limits.items():
        df[col] = df[col].clip(lo, hi)

    return df

# Generate the dataset
icu_data = generate_icu_channel_data(n_samples=500)

print(f"Dataset: {len(icu_data)} readings from 15 bedside channels (simulated, no real patient data)")
print(f"Deterioration events: {icu_data['is_anomaly'].sum()} ({icu_data['is_anomaly'].mean()*100:.1f}%)")
print("\nChannel Summary:")
display(icu_data.drop('is_anomaly', axis=1).describe().round(1))


Dataset: 500 readings from 15 bedside channels (simulated, no real patient data)
Deterioration events: 15 (3.0%)

Channel Summary:


,heart_rate,map_mmhg,spo2,resp_rate,temp_c,etco2,cvp,urine_output,lactate,glucose,gcs,fio2,peep,tidal_volume,pain_score
count,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0
mean,93.6,71.4,93.0,20.1,37.4,37.3,10.7,0.6,3.4,8.0,9.9,0.5,9.0,424.1,3.3
std,16.1,8.5,2.8,5.4,0.5,2.9,2.0,0.2,1.5,1.0,2.3,0.1,2.4,25.7,1.6
min,60.2,46.9,87.7,8.3,36.4,28.8,5.7,0.0,0.5,5.8,4.7,0.3,4.4,357.9,0.0
25%,81.0,65.0,90.6,15.8,37.0,35.0,9.1,0.4,2.1,7.1,8.1,0.4,7.0,404.2,2.2
50%,92.9,71.1,92.9,20.0,37.4,37.4,10.6,0.6,3.4,8.0,9.9,0.5,9.1,424.2,3.2
75%,105.4,78.2,95.4,24.4,37.8,39.3,12.3,0.8,4.7,8.8,11.7,0.6,11.2,444.8,4.5
max,160.1,90.9,98.3,31.1,38.4,45.8,15.4,1.2,6.3,10.5,15.0,0.8,14.0,489.4,7.3


In [6]:
# Visualize the correlation structure
def show_channel_correlations(df):
    """Show why many channels are redundant."""
    
    # Calculate correlation matrix
    channel_cols = [c for c in df.columns if c != 'is_anomaly']
    corr_matrix = df[channel_cols].corr()
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=channel_cols,
        y=channel_cols,
        colorscale='RdBu',
        zmid=0,
        text=corr_matrix.values.round(2),
        texttemplate='%{text}',
        textfont=dict(size=8),
        hoverinfo='z'
    ))
    
    fig.update_layout(
        title=dict(
            text='Channel Correlation Matrix<br><sub>Red = positive correlation, Blue = negative. Many channels measure the same physiology!</sub>',
            font=dict(size=18)
        ),
        width=900,
        height=700,
        xaxis=dict(tickangle=45)
    )
    
    fig.show()
    
    # Count highly correlated pairs
    high_corr_count = 0
    for i in range(len(channel_cols)):
        for j in range(i+1, len(channel_cols)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                high_corr_count += 1
    
    print(f"\nHighly correlated channel pairs (r > 0.7): {high_corr_count}")
    print("This redundancy is exactly what PCA exploits!")
    print("If two channels always move together, we only need to track one.")

show_channel_correlations(icu_data)


Highly correlated channel pairs (r > 0.7): 79
This redundancy is exactly what PCA exploits!
If two channels always move together, we only need to track one.


---

## 4. Compressing the Data

Now we apply PCA to compress 15 channels into fewer dimensions. The question is: how much can we compress while keeping enough information to detect anomalies?

In [7]:
# Prepare data for PCA
channel_cols = [c for c in icu_data.columns if c != 'is_anomaly']
X = icu_data[channel_cols].values

# Standardize (important: put all channels on the same scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Visualize variance explained
def show_variance_explained():
    """Show how much each component captures."""
    
    var_exp = pca.explained_variance_ratio_ * 100
    cum_var = np.cumsum(var_exp)
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            'Information in Each Component',
            'Cumulative Information Preserved'
        )
    )
    
    colors = ['#2ecc71' if i < 3 else '#95a5a6' for i in range(len(var_exp))]
    
    fig.add_trace(
        go.Bar(
            x=[f'PC{i+1}' for i in range(len(var_exp))],
            y=var_exp,
            marker_color=colors,
            text=[f'{v:.1f}%' for v in var_exp],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=[f'PC{i+1}' for i in range(len(cum_var))],
            y=cum_var,
            mode='lines+markers',
            marker=dict(size=10, color='blue'),
            line=dict(width=3),
            showlegend=False
        ),
        row=1, col=2
    )
    
    for threshold in [90, 95]:
        fig.add_hline(y=threshold, line_dash='dash', line_color='red',
                      annotation_text=f'{threshold}%', row=1, col=2)
    
    fig.update_layout(
        title=dict(text='How Much Can We Compress?', font=dict(size=20)),
        width=1100,
        height=450
    )
    
    fig.update_yaxes(title_text='Variance (%)', row=1, col=1)
    fig.update_yaxes(title_text='Cumulative (%)', range=[0, 105], row=1, col=2)
    
    fig.show()
    
    for threshold in [80, 90, 95]:
        n_components = np.argmax(cum_var >= threshold) + 1
        print(f"{threshold}% information preserved with {n_components} components (from 15 channels)")

show_variance_explained()

80% information preserved with 2 components (from 15 channels)
90% information preserved with 3 components (from 15 channels)
95% information preserved with 5 components (from 15 channels)


In [8]:
def interactive_component_explorer(n_components=3):
    """Interactive exploration of compression level."""
    
    var_exp = pca.explained_variance_ratio_ * 100
    cum_var = np.cumsum(var_exp)
    total_var = cum_var[n_components - 1]
    
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.6, 0.4],
        subplot_titles=(
            f'Using {n_components} Components',
            'Compression Summary'
        )
    )
    
    colors = ['#2ecc71' if i < n_components else '#e0e0e0' for i in range(len(var_exp))]
    
    fig.add_trace(
        go.Bar(
            x=[f'PC{i+1}' for i in range(len(var_exp))],
            y=var_exp,
            marker_color=colors,
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(
            x=['Original', 'Compressed'],
            y=[15, n_components],
            marker_color=['#3498db', '#2ecc71'],
            text=['15', f'{n_components}'],
            textposition='outside',
            textfont=dict(size=16),
            showlegend=False
        ),
        row=1, col=2
    )
    
    quality_color = '#27ae60' if total_var >= 90 else '#f39c12' if total_var >= 80 else '#e74c3c'
    
    fig.update_layout(
        title=dict(
            text=f'Compression: 15 channels → {n_components} components ({total_var:.1f}% information preserved)',
            font=dict(size=18)
        ),
        width=1100,
        height=400,
        annotations=[
            dict(
                x=0.85, y=0.5,
                xref='paper', yref='paper',
                text=f'<b>{total_var:.1f}%</b><br>preserved',
                showarrow=False,
                font=dict(size=20, color=quality_color),
                align='center'
            )
        ]
    )
    
    fig.update_yaxes(title_text='Variance (%)', row=1, col=1)
    fig.update_yaxes(title_text='Dimensions', range=[0, 18], row=1, col=2)
    
    fig.show()

interact(
    interactive_component_explorer,
    n_components=IntSlider(
        value=3, min=1, max=10, step=1,
        description='Components:',
        style={'description_width': '100px'}
    )
);

interactive(children=(IntSlider(value=3, description='Components:', max=10, min=1, style=SliderStyle(descripti…

---

## 5. The Real Value: Catching Problems Early

The power of PCA isn't in naming individual components - it's in **anomaly detection**.

By compressing 15 channels into a compact representation, we can detect when a patient stops looking like themselves.

**The Key Insight:**
- When we compress data and then reconstruct it, **readings that fit the normal pattern reconstruct well**
- **Readings that break the pattern don't** - they have high "reconstruction error"
- This reconstruction error becomes our early warning signal

**Why This Works:**
PCA learns how the channels normally move *together*: when a patient gets sicker, the heart rate rises AND the pressure falls AND the lactate climbs AND the oxygen requirement goes up, all in step. The alert that matters is a **decoupling** - the heart rate jumps 40 beats while the temperature, lactate and ventilator settings sit completely still. No single channel has breached its own alarm limit, and a patient who was merely sicker would have moved every channel at once. PCA has never seen this combination, so the reconstruction error spikes and someone goes to look.


---

## 5a. Bonus: PCA Reveals Hidden Structure

Before we get to anomaly detection, here's another powerful use of PCA: **discovering hidden categories in your data**.

The famous example: In 2008, researchers applied PCA to genetic data from Europeans. The result? PC1 and PC2 almost perfectly recreated a map of Europe - people clustered by country, and their positions matched actual geography. Genetic variation encoded geography.

**For patients:** Can PCA reveal clinical states from bedside data alone?


In [9]:
def generate_patient_states_data():
    """Generate bedside data for four distinct patient states.

    The four states - Stable, Post-operative, Sepsis, Recovering - are the same
    ones used in notebook 10, so the two PCA notebooks tell one continuous story.
    All values are simulated; no real patient data is used.
    """
    np.random.seed(42)
    n_per_state = 150

    states = []

    # State 1: Stable - everything near normal
    stable = pd.DataFrame({
        'heart_rate': np.random.normal(78, 6, n_per_state),
        'map_mmhg': np.random.normal(86, 5, n_per_state),
        'resp_rate': np.random.normal(15, 1.8, n_per_state),
        'temp_c': np.random.normal(36.8, 0.3, n_per_state),
        'lactate': np.random.normal(1.1, 0.25, n_per_state),
        'gcs': np.random.normal(15, 0.4, n_per_state),
    })
    stable['state'] = 'Stable'
    states.append(stable)

    # State 2: Post-operative - mild stress response, still partly sedated
    postop = pd.DataFrame({
        'heart_rate': np.random.normal(94, 8, n_per_state),
        'map_mmhg': np.random.normal(78, 7, n_per_state),
        'resp_rate': np.random.normal(16, 2.5, n_per_state),
        'temp_c': np.random.normal(37.2, 0.4, n_per_state),
        'lactate': np.random.normal(1.7, 0.4, n_per_state),
        'gcs': np.random.normal(11, 1.6, n_per_state),
    })
    postop['state'] = 'Post-operative'
    states.append(postop)

    # State 3: Sepsis - tachycardic, hypotensive, febrile, lactate up
    sepsis = pd.DataFrame({
        'heart_rate': np.random.normal(126, 11, n_per_state),
        'map_mmhg': np.random.normal(61, 8, n_per_state),
        'resp_rate': np.random.normal(29, 4, n_per_state),
        'temp_c': np.random.normal(38.9, 0.6, n_per_state),
        'lactate': np.random.normal(4.6, 1.2, n_per_state),
        'gcs': np.random.normal(12, 1.8, n_per_state),
    })
    sepsis['state'] = 'Sepsis'
    states.append(sepsis)

    # State 4: Recovering - settling back towards baseline
    recovering = pd.DataFrame({
        'heart_rate': np.random.normal(86, 7, n_per_state),
        'map_mmhg': np.random.normal(81, 6, n_per_state),
        'resp_rate': np.random.normal(17, 2.0, n_per_state),
        'temp_c': np.random.normal(37.0, 0.3, n_per_state),
        'lactate': np.random.normal(1.4, 0.35, n_per_state),
        'gcs': np.random.normal(14, 0.9, n_per_state),
    })
    recovering['state'] = 'Recovering'
    states.append(recovering)

    df = pd.concat(states, ignore_index=True)

    # Keep every value physiologically possible
    limits = {'heart_rate': (40, 180), 'map_mmhg': (40, 130), 'resp_rate': (6, 45),
              'temp_c': (34.0, 41.5), 'lactate': (0.4, 15.0), 'gcs': (3, 15)}
    for col, (lo, hi) in limits.items():
        df[col] = df[col].clip(lo, hi)

    return df

def show_pca_classification():
    """Show how PCA reveals patient states - like genes revealed geography."""

    # Generate data with known states
    state_data = generate_patient_states_data()

    # Apply PCA
    feature_cols = ['heart_rate', 'map_mmhg', 'resp_rate', 'temp_c', 'lactate', 'gcs']
    X_states = state_data[feature_cols].values

    scaler_states = StandardScaler()
    X_states_scaled = scaler_states.fit_transform(X_states)

    pca_states = PCA(n_components=2)
    X_states_pca = pca_states.fit_transform(X_states_scaled)

    state_data['PC1'] = X_states_pca[:, 0]
    state_data['PC2'] = X_states_pca[:, 1]

    # Create visualization
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            'PCA Reveals Patient States',
            'What Drives Each Axis?'
        ),
        column_widths=[0.6, 0.4]
    )

    # Plot 1: PCA scatter colored by state
    colors = {'Stable': '#3498db', 'Post-operative': '#9b59b6',
              'Sepsis': '#e74c3c', 'Recovering': '#2ecc71'}

    for state, color in colors.items():
        subset = state_data[state_data['state'] == state]
        fig.add_trace(
            go.Scatter(
                x=subset['PC1'],
                y=subset['PC2'],
                mode='markers',
                name=state,
                marker=dict(size=8, color=color, opacity=0.6),
            ),
            row=1, col=1
        )

    # Plot 2: What each PC captures (loadings)
    loadings = pca_states.components_

    fig.add_trace(
        go.Bar(
            y=feature_cols,
            x=loadings[0],
            orientation='h',
            name='PC1',
            marker_color='#3498db',
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text='PCA for Classification: Patient States Emerge Naturally',
            font=dict(size=18)
        ),
        width=1200,
        height=500
    )

    fig.update_xaxes(title_text='PC1 (Illness Severity)', row=1, col=1)
    fig.update_yaxes(title_text='PC2 (Conscious Level / Sedation)', row=1, col=1)
    fig.update_xaxes(title_text='Loading on PC1', row=1, col=2)

    fig.show()

    print("Key Insight:")
    print("  - We didn't tell PCA about the patient states")
    print("  - It found them automatically from the bedside channels")
    print("  - PC1 ≈ 'how sick is this patient'")
    print("  - PC2 ≈ 'how awake is this patient'")
    print("\nThis is the Novembre 'genes = geography' insight for the bedside:")
    print("  Monitoring patterns encode clinical state.")

show_pca_classification()


Key Insight:
  - We didn't tell PCA about the patient states
  - It found them automatically from the bedside channels
  - PC1 ≈ 'how sick is this patient'
  - PC2 ≈ 'how awake is this patient'

This is the Novembre 'genes = geography' insight for the bedside:
  Monitoring patterns encode clinical state.


---

## 6. Anomaly Detection in Action

Let's see this in practice. We'll compress our 15 channels, reconstruct them, and measure the error. High error = the patient no longer looks like themselves.


In [10]:
def anomaly_detection_demo():
    """Demonstrate PCA-based anomaly detection."""
    
    # Use 3 components for reconstruction
    n_components = 3
    pca_reduced = PCA(n_components=n_components)
    X_reduced = pca_reduced.fit_transform(X_scaled)
    
    # Reconstruct data from reduced components
    X_reconstructed = pca_reduced.inverse_transform(X_reduced)
    
    # Calculate reconstruction error for each sample
    reconstruction_error = np.sum((X_scaled - X_reconstructed) ** 2, axis=1)
    
    # Add to dataframe
    icu_data['reconstruction_error'] = reconstruction_error
    icu_data['reading_number'] = range(len(icu_data))
    
    # Determine threshold (e.g., 95th percentile)
    threshold = np.percentile(reconstruction_error, 95)
    
    # Create visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Deviation Score Over Time',
            'Compressed Data View (PC1 vs PC2)',
            'Detection Performance',
            'Illustrative Cost per Event'
        )
    )
    
    # Plot 1: Reconstruction error time series
    fig.add_trace(
        go.Scatter(
            x=icu_data['reading_number'],
            y=icu_data['reconstruction_error'],
            mode='lines',
            line=dict(color='blue', width=1),
            name='Deviation Score',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Highlight anomalies
    anomaly_mask = icu_data['is_anomaly']
    fig.add_trace(
        go.Scatter(
            x=icu_data.loc[anomaly_mask, 'reading_number'],
            y=icu_data.loc[anomaly_mask, 'reconstruction_error'],
            mode='markers',
            marker=dict(size=10, color='red', symbol='x'),
            name='Actual Deterioration Events',
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.add_hline(y=threshold, line_dash='dash', line_color='red',
                  annotation_text='Alert Threshold', row=1, col=1)
    
    # Plot 2: PCA space scatter
    colors = ['red' if a else 'blue' for a in icu_data['is_anomaly']]
    fig.add_trace(
        go.Scatter(
            x=X_pca[:, 0],
            y=X_pca[:, 1],
            mode='markers',
            marker=dict(size=5, color=colors, opacity=0.5),
            showlegend=False
        ),
        row=1, col=2
    )
    
    # Plot 3: Detection performance
    detected = reconstruction_error > threshold
    tp = np.sum(detected & anomaly_mask)
    fn = np.sum(~detected & anomaly_mask)
    fp = np.sum(detected & ~anomaly_mask)
    tn = np.sum(~detected & ~anomaly_mask)
    
    detection_rate = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    false_alarm_rate = fp / (fp + tn) * 100 if (fp + tn) > 0 else 0
    
    fig.add_trace(
        go.Bar(
            x=['Anomalies Caught', 'False Alarms'],
            y=[detection_rate, false_alarm_rate],
            marker_color=['#2ecc71', '#e74c3c'],
            text=[f'{detection_rate:.0f}%', f'{false_alarm_rate:.1f}%'],
            textposition='outside',
            showlegend=False
        ),
        row=2, col=1
    )
    
    # Plot 4: Cost comparison
    costs = ['Unplanned\nDowntime', 'Early\nIntervention', 'Net Savings']
    values = [50000, 5000, 45000]
    cost_colors = ['#e74c3c', '#f39c12', '#2ecc71']
    
    fig.add_trace(
        go.Bar(
            x=costs,
            y=values,
            marker_color=cost_colors,
            text=['$50K', '$5K', '$45K'],
            textposition='outside',
            showlegend=False
        ),
        row=2, col=2
    )
    
    fig.update_layout(
        title=dict(text='PCA Deterioration Detection: From 15 Channels to 1 Alert', font=dict(size=20)),
        width=1200,
        height=700
    )
    
    fig.update_xaxes(title_text='Reading Number', row=1, col=1)
    fig.update_yaxes(title_text='Deviation Score', row=1, col=1)
    fig.update_xaxes(title_text='PC1', row=1, col=2)
    fig.update_yaxes(title_text='PC2', row=1, col=2)
    fig.update_yaxes(title_text='Rate (%)', range=[0, 110], row=2, col=1)
    fig.update_yaxes(title_text='Cost ($)', row=2, col=2)
    
    fig.show()
    
    print(f"\nDeterioration Detection Results:")
    print(f"  True Deterioration Events in Data: {int(tp + fn)}")
    print(f"  Caught by PCA: {int(tp)} ({detection_rate:.0f}%)")
    print(f"  False Alarms: {int(fp)} ({false_alarm_rate:.1f}%)")
    print(f"\nOne number to watch. Early warning. Actionable.")

anomaly_detection_demo()


Deterioration Detection Results:
  True Deterioration Events in Data: 15
  Caught by PCA: 15 (100%)
  False Alarms: 10 (2.1%)

One number to watch. Early warning. Actionable.


In [11]:
def interactive_threshold_explorer(sensitivity=2.0):
    """Interactive exploration of detection sensitivity."""
    
    # Use reconstruction error
    pca_3 = PCA(n_components=3)
    X_reduced = pca_3.fit_transform(X_scaled)
    X_reconstructed = pca_3.inverse_transform(X_reduced)
    reconstruction_error = np.sum((X_scaled - X_reconstructed) ** 2, axis=1)
    
    # Calculate threshold based on sensitivity
    mean_error = np.mean(reconstruction_error)
    std_error = np.std(reconstruction_error)
    threshold = mean_error + sensitivity * std_error
    
    # Detect anomalies
    detected = reconstruction_error > threshold
    actual = icu_data['is_anomaly'].values
    
    # Calculate metrics
    tp = np.sum(detected & actual)
    fn = np.sum(~detected & actual)
    fp = np.sum(detected & ~actual)
    
    detection_rate = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    n_alerts = np.sum(detected)
    
    # Create visualization
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.7, 0.3],
        subplot_titles=(
            f'Deviation from Normal (Sensitivity: {sensitivity:.1f})',
            'Detection Summary'
        )
    )
    
    # Time series with threshold
    fig.add_trace(
        go.Scatter(
            x=list(range(len(reconstruction_error))),
            y=reconstruction_error,
            mode='lines',
            line=dict(color='blue', width=1),
            name='Deviation',
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.add_hline(
        y=threshold,
        line_dash='dash',
        line_color='red',
        row=1, col=1
    )
    
    # Summary metrics
    fig.add_trace(
        go.Bar(
            x=['Total Alerts', 'True Catches', 'False Alarms'],
            y=[n_alerts, tp, fp],
            marker_color=['#3498db', '#2ecc71', '#e74c3c'],
            text=[f'{n_alerts}', f'{int(tp)}', f'{int(fp)}'],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title=dict(
            text=f'Sensitivity Trade-off: {detection_rate:.0f}% of anomalies detected with {int(fp)} false alarms',
            font=dict(size=16)
        ),
        width=1100,
        height=400
    )
    
    fig.update_yaxes(title_text='Deviation Score', row=1, col=1)
    fig.update_yaxes(range=[0, max(n_alerts + 5, 30)], row=1, col=2)
    
    fig.show()

# Create interactive widget
interact(
    interactive_threshold_explorer,
    sensitivity=FloatSlider(
        value=2.0,
        min=0.5,
        max=3.5,
        step=0.25,
        description='Sensitivity:',
        style={'description_width': '100px'}
    )
);

interactive(children=(FloatSlider(value=2.0, description='Sensitivity:', max=3.5, min=0.5, step=0.25, style=Sl…

**Key Insight:** There's always a trade-off between catching more deteriorations and generating more false alarms. The deviation score gives you one number to tune - much simpler, and far less alarm-fatiguing, than setting limits on 15 individual channels.

---

## 7. From One Bed to the Whole Ward

The same approach scales across the unit. Each bed gets one deviation score, so the charge nurse sees the whole ward on a single screen.


In [12]:
def create_ward_dashboard():
    """Create a ward-wide monitoring dashboard based on deviation scores."""

    # Simulate a 12-bed ICU, one deviation score per bed
    np.random.seed(123)

    beds = [
        ('ICU-01', 0.3, 'Normal'),
        ('ICU-02', 0.5, 'Normal'),
        ('ICU-03', 1.8, 'Attention'),
        ('ICU-04', 0.4, 'Normal'),
        ('ICU-05', 3.2, 'Alert'),
        ('ICU-06', 0.7, 'Normal'),
        ('ICU-07', 0.2, 'Normal'),
        ('ICU-08', 2.1, 'Attention'),
        ('ICU-09', 0.6, 'Normal'),
        ('ICU-10', 0.9, 'Normal'),
        ('ICU-11', 2.8, 'Alert'),
        ('ICU-12', 1.1, 'Normal'),
    ]

    # Create figure
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.6, 0.4],
        subplot_titles=(
            'Ward Deviation Scores',
            'Status Summary'
        )
    )

    # Ward deviation scores - horizontal bar chart
    bed_names = [b[0] for b in beds]
    scores = [b[1] for b in beds]
    status_colors = ['#2ecc71' if s == 'Normal' else '#f39c12' if s == 'Attention' else '#e74c3c'
                     for _, _, s in beds]

    fig.add_trace(
        go.Bar(
            y=bed_names[::-1],
            x=scores[::-1],
            orientation='h',
            marker_color=status_colors[::-1],
            text=[f'{s:.1f}' for s in scores[::-1]],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=1
    )

    # Add threshold lines
    fig.add_vline(x=1.5, line_dash='dash', line_color='orange',
                  annotation_text='Attention', row=1, col=1)
    fig.add_vline(x=2.5, line_dash='dash', line_color='red',
                  annotation_text='Alert', row=1, col=1)

    # Status summary
    status_counts = {'Normal': 0, 'Attention': 0, 'Alert': 0}
    for _, _, status in beds:
        status_counts[status] += 1

    fig.add_trace(
        go.Bar(
            x=list(status_counts.keys()),
            y=list(status_counts.values()),
            marker_color=['#2ecc71', '#f39c12', '#e74c3c'],
            text=list(status_counts.values()),
            textposition='outside',
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text='Ward Monitoring: One Deviation Score Per Bed',
            font=dict(size=20)
        ),
        width=1100,
        height=550
    )

    fig.update_xaxes(title_text='Deviation Score', range=[0, 4], row=1, col=1)
    fig.update_yaxes(title_text='Bed Count', row=1, col=2)

    fig.show()

    print("\nWard Status:")
    print("=" * 50)
    for bed, score, status in beds:
        print(f"  {bed}: {score:.1f}  ({status})")

    print("\nWith PCA-based monitoring:")
    print("  - 1 number per bed (not 15+ channels)")
    print("  - Instant ward-wide view for the charge nurse")
    print("  - Clear thresholds for escalation")
    print("\nThe score is decision support: it says 'go and look at this patient',")
    print("never 'here is the diagnosis'.")

create_ward_dashboard()



Ward Status:
  ICU-01: 0.3  (Normal)
  ICU-02: 0.5  (Normal)
  ICU-03: 1.8  (Attention)
  ICU-04: 0.4  (Normal)
  ICU-05: 3.2  (Alert)
  ICU-06: 0.7  (Normal)
  ICU-07: 0.2  (Normal)
  ICU-08: 2.1  (Attention)
  ICU-09: 0.6  (Normal)
  ICU-10: 0.9  (Normal)
  ICU-11: 2.8  (Alert)
  ICU-12: 1.1  (Normal)

With PCA-based monitoring:
  - 1 number per bed (not 15+ channels)
  - Instant ward-wide view for the charge nurse
  - Clear thresholds for escalation

The score is decision support: it says 'go and look at this patient',
never 'here is the diagnosis'.


---

## 8. The Bottom Line

### What PCA Gives You

| Before | After |
|--------|-------|
| 15+ channels per bed | 1 deviation score |
| Complex per-channel alarm limits | Single alert threshold |
| Missed patterns across channels | Catches multi-channel deterioration |
| Per-channel alarms and alarm fatigue | Ward-wide dashboard |

### The Core Idea

1. **Compress**: PCA learns what a stable patient's channels look like together and compresses 15 channels into fewer dimensions
2. **Reconstruct**: We can approximate the original values from the compressed version
3. **Measure deviation**: A stable patient reconstructs well. A deteriorating one doesn't.
4. **Alert**: High reconstruction error = go and review this patient

### What PCA Doesn't Do

- It doesn't tell you *which* channel is the problem (the clinician drills down after the alert)
- The components aren't interpretable as "cardiac health" or "respiratory status"
- It's a compression technique, not a diagnosis

### Clinical Governance

- The score is **decision support**, not a diagnosis, and never an automatic treatment decision
- Deviation is measured against *the population it was fitted on* - refit for your case mix
- A missed deterioration is far costlier than a false alarm; choose the threshold accordingly, with clinicians in the room
- Audit the score against outcomes before, and after, it goes live

### Strategic Value

- **Early warning**: Catch deterioration before any single channel breaches its own limit
- **Simplicity**: One number per bed, clear escalation thresholds
- **Scalability**: Same approach works for 12 beds or 1,200
- **Foundation**: First step toward a full early-warning system


In [13]:
def create_summary_infographic():
    """Create final summary visualization."""
    
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            'BEFORE: 15 channels',
            'PCA Compression',
            'AFTER: 1 Deviation Score'
        )
    )
    
    # Before: Many channels
    x_before = np.tile([0, 1, 2], 5)
    y_before = np.repeat([0, 1, 2, 3, 4], 3)
    
    fig.add_trace(
        go.Scatter(
            x=x_before, y=y_before,
            mode='markers',
            marker=dict(size=30, color='#3498db', line=dict(color='black', width=1)),
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Middle: Arrow
    fig.add_trace(
        go.Scatter(
            x=[0, 2], y=[2, 2],
            mode='lines+text',
            line=dict(color='#2ecc71', width=5),
            text=['', 'PCA'],
            textposition='top center',
            textfont=dict(size=20, color='#2ecc71'),
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.add_annotation(
        x=1, y=1.5,
        text='Learn normal patterns<br>Detect deviations',
        showarrow=False,
        font=dict(size=12),
        xref='x2', yref='y2'
    )
    
    # After: Single score
    fig.add_trace(
        go.Scatter(
            x=[1], y=[2],
            mode='markers+text',
            marker=dict(size=100, color='#2ecc71', line=dict(color='black', width=2)),
            text=['0.3'],
            textfont=dict(size=24, color='white'),
            showlegend=False
        ),
        row=1, col=3
    )
    
    fig.add_annotation(
        x=1, y=0.8,
        text='Normal',
        showarrow=False,
        font=dict(size=16, color='#2ecc71'),
        xref='x3', yref='y3'
    )
    
    fig.update_layout(
        title=dict(text='PCA for early deterioration scoring: The Simple Version', font=dict(size=22)),
        width=1200,
        height=400
    )
    
    for i in range(1, 4):
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
    
    fig.show()
    
    print("\n" + "="*60)
    print("KEY TAKEAWAYS")
    print("="*60)
    print("\n1. PCA compresses many channels into a compact form")
    print("2. Reconstruction error = how 'abnormal' a reading is")
    print("3. One deviation score per bed, one threshold to set")
    print("4. Catches multi-channel anomalies that individual thresholds miss")
    print("5. Foundation for ward-wide early deterioration scoring")
    print("\n" + "="*60)

create_summary_infographic()


KEY TAKEAWAYS

1. PCA compresses many channels into a compact form
2. Reconstruction error = how 'abnormal' a reading is
3. One deviation score per bed, one threshold to set
4. Catches multi-channel anomalies that individual thresholds miss
5. Foundation for ward-wide early deterioration scoring



In [14]:
print("\nPCA Executive Demonstration Complete")
print("\nQuestions for Discussion:")
print("  - What monitoring and lab data do you already capture electronically?")
print("  - How is deterioration currently recognized on your wards?")
print("  - What is the cost of a missed deterioration vs. an extra review?")
print("  - How would a single deviation score per bed change your handover?")



PCA Executive Demonstration Complete

Questions for Discussion:
  - What monitoring and lab data do you already capture electronically?
  - How is deterioration currently recognized on your wards?
  - What is the cost of a missed deterioration vs. an extra review?
  - How would a single deviation score per bed change your handover?
